<a href="https://colab.research.google.com/github/erickodiwuor014-max/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Money Agent Network: Data Audit and Cleaning
Engagement 2 of `plp-da-a3-duka-sales-and-agent-cleanup`. Data: `agent_transactions.csv` and `agents.csv`, generated by `generate_data.py` (seed 44). The cleaning rules live in `cleaning_pipeline.py` and are applied by calling `clean_agent_data()`.

In [1]:
import os, sys, subprocess

REPO = "plp-da-a3-duka-sales-and-agent-cleanup"
if not os.path.exists("generate_data.py"):
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/erickodiwuor014-max/{REPO}.git"], check=True)
    os.chdir(REPO)
if not os.path.exists("agent_transactions.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

print("Working folder:", os.getcwd())
print("Files:", sorted(os.listdir()))

Working folder: /content/plp-da-a3-duka-sales-and-agent-cleanup
Files: ['.git', '.gitignore', 'README.md', '__pycache__', 'agent_transactions.csv', 'agents.csv', 'cleaning_pipeline.py', 'duka_sales.csv', 'duka_sales_exploration.ipynb', 'excel', 'generate_data.py', 'outputs', 'requirements.txt']


## 1. Raw data inspection
The raw files are loaded exactly as they arrive, with no cleaning, so every injected problem can be found and counted.

In [2]:
raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")
print("Transactions:", raw_txns.shape)
print("Agents:", raw_agents.shape)
raw_txns.head(10)

Transactions: (612, 5)
Agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17
5,TX00006,AG019,withdrawal,931.0,2026-04-23
6,TX00007,AG020,deposit,2583.0,2026-04-02
7,TX00008,AG019,deposit,1797.0,2026-04-12
8,TX00009,AG012,Withdrawal,4184.0,2026-04-04
9,TX00010,AG015,DEPOSIT,1987.0,2026-04-17


In [3]:
raw_txns.info()
print()
print("Missing values per column:")
print(raw_txns.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB

Missing values per column:
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


In [4]:
print("Duplicate full rows:", raw_txns.duplicated().sum())
print("Duplicate txn_id values:", raw_txns["txn_id"].duplicated().sum())
print()
print("Distinct txn_type values (shown with quotes to reveal spaces):")
print(raw_txns["txn_type"].map(repr).value_counts())

Duplicate full rows: 11
Duplicate txn_id values: 12

Distinct txn_type values (shown with quotes to reveal spaces):
txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100
Name: count, dtype: int64


In [5]:
print("Amount values that are not numbers or blanks:")
amt_num = pd.to_numeric(raw_txns["amount"], errors="coerce")
print(raw_txns.loc[amt_num.isna() & raw_txns["amount"].notna(), "amount"].value_counts())
print()
print("Amount description:")
print(amt_num.describe())
print()
print("Rows with amount exactly 250000:", (amt_num == 250000).sum())

Amount values that are not numbers or blanks:
Series([], Name: count, dtype: int64)

Amount description:
count       590.000000
mean       4537.201695
std       22836.165375
min         109.000000
25%        1086.750000
50%        1779.000000
75%        3069.750000
max      250000.000000
Name: amount, dtype: float64

Rows with amount exactly 250000: 5


In [6]:
slash = raw_txns["txn_date"].str.contains("/", na=False)
print("Dates written with slashes:", slash.sum())
print(raw_txns.loc[slash, "txn_date"].head())
print()
parsed = pd.to_datetime(raw_txns["txn_date"].str.replace("/", "-"), errors="coerce")
print("Date range:", parsed.min().date(), "to", parsed.max().date())
print("Dates that still fail to parse:", parsed.isna().sum())

Dates written with slashes: 5
87     2026/04/27
128    2026/04/28
181    2026/03/20
437    2026/03/30
571    2026/03/04
Name: txn_date, dtype: object

Date range: 2026-03-01 to 2026-04-29
Dates that still fail to parse: 0


In [7]:
unknown = ~raw_txns["agent_id"].isin(raw_agents["agent_id"])
print("Transactions with agent_id not in the register:", unknown.sum())
print(raw_txns.loc[unknown, "agent_id"].value_counts())
print()
print(raw_agents)

Transactions with agent_id not in the register: 8
agent_id
AG099    8
Name: count, dtype: int64

   agent_id   county   onboarded
0     AG001  Mombasa  2024-01-01
1     AG002  Nairobi  2024-02-01
2     AG003   Kisumu  2024-03-01
3     AG004   Kiambu  2024-04-01
4     AG005  Mombasa  2024-05-01
5     AG006   Nakuru  2024-06-01
6     AG007  Nairobi  2024-07-01
7     AG008   Kisumu  2024-08-01
8     AG009   Nakuru  2024-09-01
9     AG010  Nairobi  2024-10-01
10    AG011   Kisumu  2024-11-01
11    AG012   Kisumu  2024-12-01
12    AG013  Nairobi  2025-01-01
13    AG014  Nairobi  2025-02-01
14    AG015  Nairobi  2025-03-01
15    AG016   Kiambu  2025-04-01
16    AG017   Nakuru  2025-05-01
17    AG018  Mombasa  2025-06-01
18    AG019  Mombasa  2025-07-01
19    AG020  Mombasa  2025-08-01


## 2. Data-quality audit
Every problem found in the raw data, counted and classified by dimension. Counts come from code run on the raw files, not typed in.

In [24]:
raw_txt = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)
amt_num = pd.to_numeric(raw_txns["amount"], errors="coerce")

q1, q3 = amt_num.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

parsed = pd.to_datetime(raw_txns["txn_date"].str.replace("/", "-"), errors="coerce")
onboard = pd.to_datetime(raw_txns["agent_id"].map(raw_agents.set_index("agent_id")["onboarded"]))
timeliness = ((parsed < onboard) | (parsed > pd.Timestamp.today())).sum()

audit = pd.DataFrame([
    ("Missing amount (blank)", "Completeness", "amount", (raw_txt["amount"] == "").sum()),
    ("Invalid text in amount ('N/A')", "Validity", "amount", (raw_txt["amount"] == "N/A").sum()),
    ("Duplicate transactions (repeated txn_id)", "Uniqueness", "txn_id", raw_txns["txn_id"].duplicated().sum()),
    ("txn_type not exactly Deposit or Withdrawal (case, trailing space)", "Consistency", "txn_type",
     (~raw_txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Dates written with slashes instead of ISO dashes", "Consistency", "txn_date",
     raw_txns["txn_date"].str.contains("/").sum()),
    ("agent_id not in the agent register (AG099 - 8 raw, 7 after deduplication)", "Validity", "agent_id",
     (~raw_txns["agent_id"].isin(raw_agents["agent_id"])).sum()),
    ("Repeated extreme amount of exactly 250,000", "Accuracy", "amount", (amt_num == 250000).sum()),
    (f"Statistical outliers by IQR rule (outside {low:,.0f} to {high:,.0f})", "Accuracy", "amount",
     ((amt_num < low) | (amt_num > high)).sum()),
    ("Transactions dated before the agent was onboarded or in the future", "Timeliness", "txn_date", timeliness),
], columns=["issue", "dimension", "column", "rows_affected"])

audit

,issue,dimension,column,rows_affected
0,Missing amount (blank),Completeness,amount,15
1,Invalid text in amount ('N/A'),Validity,amount,7
2,Duplicate transactions (repeated txn_id),Uniqueness,txn_id,12
3,txn_type not exactly Deposit or Withdrawal (ca...,Consistency,txn_type,384
4,Dates written with slashes instead of ISO dashes,Consistency,txn_date,5
5,agent_id not in the agent register (AG099 - 8 ...,Validity,agent_id,8
6,"Repeated extreme amount of exactly 250,000",Accuracy,amount,5
7,"Statistical outliers by IQR rule (outside -1,8...",Accuracy,amount,40
8,Transactions dated before the agent was onboar...,Timeliness,txn_date,0


**Reading the audit.** Pandas reads the text "N/A" as missing by default, so the blank and "N/A" rows together make the 22 missing amounts seen in `isna()`. The 12 repeated `txn_id` values include 11 exact copies and one copy whose `agent_id` differs, so duplicates are removed on `txn_id`, the unique key, not on whole rows. The IQR row counts every high value, including the 250,000 rows, so it is larger than the 250,000 row: a long-tailed amount distribution produces legitimate large values too. All dates fall between 1 March and 29 April 2026 and none precede an agent's onboarding date.

In [9]:
%cd /content
!rm -rf plp-da-a3-duka-sales-and-agent-cleanup
!git clone https://github.com/erickodiwuor014-max/plp-da-a3-duka-sales-and-agent-cleanup.git
%cd plp-da-a3-duka-sales-and-agent-cleanup
!python generate_data.py
!ls

/content
Cloning into 'plp-da-a3-duka-sales-and-agent-cleanup'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 53 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 97.45 KiB | 2.03 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/plp-da-a3-duka-sales-and-agent-cleanup
agents.csv		duka_sales_exploration.ipynb  README.md
agent_transactions.csv	excel			      requirements.txt
cleaning_pipeline.py	generate_data.py
duka_sales.csv		outputs


In [10]:
%cd /content
!rm -rf plp-da-a3-duka-sales-and-agent-cleanup
!git clone https://github.com/erickodiwuor014-max/plp-da-a3-duka-sales-and-agent-cleanup.git
%cd plp-da-a3-duka-sales-and-agent-cleanup
!python generate_data.py
!ls

/content
Cloning into 'plp-da-a3-duka-sales-and-agent-cleanup'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 53 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 97.45 KiB | 2.50 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/plp-da-a3-duka-sales-and-agent-cleanup
agents.csv		duka_sales_exploration.ipynb  README.md
agent_transactions.csv	excel			      requirements.txt
cleaning_pipeline.py	generate_data.py
duka_sales.csv		outputs


In [11]:
import sys
import pandas as pd
sys.path.insert(0, "/content/plp-da-a3-duka-sales-and-agent-cleanup")

raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")
print(raw_txns.shape, raw_agents.shape)

(612, 5) (20, 3)


In [12]:
from cleaning_pipeline import clean_agent_data

clean_df, log = clean_agent_data(raw_txns, raw_agents)
display(log)
print("Shape:", clean_df.shape)
clean_df.head()

,step,dimension,column,issue,rows_affected,action
0,1,Consistency,txn_type,Mixed case and trailing spaces,384,"Stripped spaces, lower-cased, mapped to Deposi..."
1,1,Validity,txn_type,Unrecognised transaction type,0,Left as missing (none expected)
2,2,Consistency,txn_date,Dates written with slashes,5,Replaced / with - and parsed to datetime
3,2,Validity,txn_date,Dates that could not be parsed,0,Left as missing (none expected)
4,3,Validity,amount,Text in amount (for example 'N/A'),0,Converted to numeric; invalid text became missing
5,3,Completeness,amount,Missing amounts (blank or N/A),22,Imputed in step 6 and flagged amount_imputed
6,4,Uniqueness,txn_id,Duplicate transactions (repeated txn_id),12,"Kept one row per txn_id, preferring the copy w..."
7,5,Validity,agent_id,agent_id not in the agent register,7,"Kept (the money moved), county set to 'Unknown..."
8,5,Timeliness,txn_date,Transactions dated before the agent was onboarded,0,Checked only (none expected)
9,6,Accuracy,amount,"IQR outliers (fences -1,928 to 6,084)",40,"Flagged is_outlier, not removed"


Shape: (600, 9)


,txn_id,agent_id,county,txn_type,amount,txn_date,amount_imputed,is_outlier,unknown_agent
0,TX00001,AG009,Nakuru,Deposit,353.0,2026-03-03,False,False,False
1,TX00002,AG020,Mombasa,Deposit,1238.0,2026-04-27,False,False,False
2,TX00003,AG008,Kisumu,Deposit,9129.0,2026-04-29,False,True,False
3,TX00004,AG009,Nakuru,Deposit,652.0,2026-04-24,False,False,False
4,TX00005,AG009,Nakuru,Withdrawal,478.0,2026-04-17,False,False,False


In [13]:
print("Rows before / after:", len(raw_txns), "/", len(clean_df))
print("Duplicate txn_id:", clean_df["txn_id"].duplicated().sum())
print()
print(clean_df["txn_type"].value_counts())
print()
print(clean_df.dtypes)
print()
print(clean_df[["amount_imputed", "is_outlier", "unknown_agent"]].sum())

Rows before / after: 612 / 600
Duplicate txn_id: 0

txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

txn_id                    object
agent_id                  object
county                    object
txn_type                  object
amount                   float64
txn_date          datetime64[ns]
amount_imputed              bool
is_outlier                  bool
unknown_agent               bool
dtype: object

amount_imputed    20
is_outlier        40
unknown_agent      7
dtype: int64


In [14]:
raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")

In [15]:
import os, sys, subprocess

REPO = "plp-da-a3-duka-sales-and-agent-cleanup"
if not os.path.exists("generate_data.py"):
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/erickodiwuor014-max/{REPO}.git"], check=True)
    os.chdir(REPO)
if not os.path.exists("agent_transactions.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

raw_txns = pd.read_csv("agent_transactions.csv")
raw_agents = pd.read_csv("agents.csv")
clean_df, log = clean_agent_data(raw_txns, raw_agents)

print("Working folder:", os.getcwd())
print("Raw:", raw_txns.shape, "| Clean:", clean_df.shape)

Working folder: /content/plp-da-a3-duka-sales-and-agent-cleanup
Raw: (612, 5) | Clean: (600, 9)


## 4. Merge validation
The pipeline joins transactions to the agent register with a **left join**, so no transaction is lost when its agent is missing from the register. An inner join would silently drop those rows and understate agent-network volume.

In [16]:
inner_rows = len(clean_df.merge(raw_agents[["agent_id"]], on="agent_id", how="inner"))
unknown = clean_df[clean_df["unknown_agent"]]

print("Transactions before merge (after dedup):", len(clean_df))
print("Transactions after the left join:       ", len(clean_df))
print("Rows an inner join would have kept:     ", inner_rows)
print()
print("Transactions with unknown agents:", len(unknown))
print("Agent ids involved:", unknown["agent_id"].unique().tolist())
print("Value of those transactions (KES):", f"{unknown['amount'].sum():,.0f}")
print(f"Share of total value: {unknown['amount'].sum() / clean_df['amount'].sum() * 100:.1f}%")
print()
print(clean_df["county"].value_counts())

Transactions before merge (after dedup): 600
Transactions after the left join:        600
Rows an inner join would have kept:      593

Transactions with unknown agents: 7
Agent ids involved: ['AG099']
Value of those transactions (KES): 27,156
Share of total value: 1.0%

county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


**Findings (merge).** The left join kept all **600** transactions, so none were lost or duplicated. An inner join would have kept only **593** and silently dropped 7 transactions worth **KES 27,156** (1.0% of total value). All 7 carry the agent ID **AG099**, which is not in the register of 20 agents (AG001 to AG020). The raw file had 8 AG099 rows, and one was a duplicate copy removed during deduplication.

**Treatment:** the 7 rows are kept because the money moved, their county is set to "Unknown", and they are flagged `unknown_agent = True`. They count in network totals but stay separate in county summaries. AG099 may be a data-entry error, an unregistered agent or an out-of-date register, so the agent-management team should confirm it.

In [17]:
big = clean_df[clean_df["amount"] == 250000]
display(big)

next_max = clean_df.loc[clean_df["amount"] < 250000, "amount"].max()
median_amt = clean_df.loc[~clean_df["amount_imputed"], "amount"].median()
print("Rows at exactly 250,000:", len(big))
print("Next largest amount:", f"{next_max:,.0f}")
print("Median amount:", f"{median_amt:,.0f}", "-> 250,000 is", f"{250000 / median_amt:,.0f}", "times the median")
print()
print("Other flagged outliers (excluding the 250,000 rows):",
      (clean_df["is_outlier"] & (clean_df["amount"] != 250000)).sum())
print()
print("Average amount with all rows:      ", f"{clean_df['amount'].mean():,.0f}")
print("Average amount without 250,000 rows:", f"{clean_df.loc[clean_df['amount'] != 250000, 'amount'].mean():,.0f}")

,txn_id,agent_id,county,txn_type,amount,txn_date,amount_imputed,is_outlier,unknown_agent
306,TX00307,AG012,Kisumu,Deposit,250000.0,2026-03-02,False,True,False
315,TX00316,AG014,Nairobi,Deposit,250000.0,2026-04-01,False,True,False
424,TX00425,AG005,Mombasa,Withdrawal,250000.0,2026-03-02,False,True,False
463,TX00464,AG009,Nakuru,Withdrawal,250000.0,2026-04-11,False,True,False
594,TX00595,AG016,Kiambu,Deposit,250000.0,2026-03-11,False,True,False


Rows at exactly 250,000: 5
Next largest amount: 28,847
Median amount: 1,776 -> 250,000 is 141 times the median

Other flagged outliers (excluding the 250,000 rows): 35

Average amount with all rows:       4,481
Average amount without 250,000 rows: 2,418


**Findings (outliers).** Exactly five transactions have an amount of KES 250,000, one in each county, from five different agents (three deposits and two withdrawals). The next largest amount in the whole file is KES 28,847, and the median is KES 1,776, so each of these is **141 times the median**. Together they are KES 1,250,000, or 46.5% of all transaction value. They lift the average transaction from **KES 2,418 to KES 4,481**. Two of them occurred on the same day (2 March 2026, in Kisumu and Mombasa). The IQR rule flagged 40 amounts in total, but the other 35 are large and plausible, with a maximum of KES 28,847, and are kept as genuine.

**Recommendation.**
1. **Do not delete them.** They may be real, so they stay in the clean file with `is_outlier = True`.
2. **Verify before use.** Check the five transaction IDs (TX00307, TX00316, TX00425, TX00464, TX00595) against the legacy system and the agents' records. Identical round amounts across five agents point to a default value, a test entry or a system limit.
3. **Report two views** until they are confirmed: totals with the five rows, and totals and averages without them. Use the median for "typical" values.
4. **If unconfirmed**, exclude them from agent and county performance comparisons. If confirmed, keep them and apply any large-value compliance checks.

In [18]:
county_summary = (
    clean_df.groupby(["county", "txn_type"])["amount"]
    .agg(total_value="sum", transactions="count", average_value="mean")
    .round(0)
    .reset_index()
)
display(county_summary)

by_county = (
    clean_df.groupby("county")["amount"]
    .agg(total_value="sum", transactions="count", average_value="mean")
    .round(0).sort_values("total_value", ascending=False)
)
by_county["share_of_value_pct"] = (by_county["total_value"] / by_county["total_value"].sum() * 100).round(1)
display(by_county)

,county,txn_type,total_value,transactions,average_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,77024.0,27,2853.0
2,Kisumu,Deposit,436503.0,74,5899.0
3,Kisumu,Withdrawal,76348.0,37,2063.0
4,Mombasa,Deposit,149096.0,72,2071.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,543700.0,118,4608.0
7,Nairobi,Withdrawal,161930.0,62,2612.0
8,Nakuru,Deposit,184945.0,60,3082.0
9,Nakuru,Withdrawal,328568.0,39,8425.0


,total_value,transactions,average_value,share_of_value_pct
county,,,,
Nairobi,705630.0,180,3920.0,26.2
Mombasa,517688.0,133,3892.0,19.3
Nakuru,513513.0,99,5187.0,19.1
Kisumu,512851.0,111,4620.0,19.1
Kiambu,411761.0,70,5882.0,15.3
Unknown,27156.0,7,3879.0,1.0


**Findings (county summary).**
- **Nairobi leads** on both value (KES 705,630, 26.2% of the total) and volume (180 transactions, 30% of the 600). It also has the most deposits: 118 worth KES 543,700.
- **Mombasa** is second (KES 517,688, 19.3%). **Nakuru** (KES 513,513) and **Kisumu** (KES 512,851) are essentially tied at 19.1%, a gap of only KES 662, though Kisumu handles more transactions (111 against 99). **Kiambu** is last on both value (KES 411,761, 15.3%) and volume (70).
- Deposits make up 62% of both volume (373 of 600) and value (KES 1,674,260 of 2,688,599).
- **Average values are distorted.** Kiambu's average of KES 5,882 and Nakuru's of KES 5,187 look highest, but each includes one KES 250,000 transaction. Without it they fall to about KES 2,344 and KES 2,689.
- **Sensitivity:** excluding the five 250,000 rows and the 20 imputed amounts removes between 37% (Nairobi) and 61% (Kiambu) of each county's value, but the ranking stays the same (Nairobi, Mombasa, Nakuru, Kisumu, Kiambu). The ranking is therefore reliable, while the exact totals and averages are not. One transaction makes up 60.7% of Kiambu's value.

In [19]:
clean_only = clean_df[~clean_df["amount_imputed"] & (clean_df["amount"] != 250000)]
compare = pd.DataFrame({
    "all_rows": clean_df.groupby("county")["amount"].sum(),
    "without_imputed_and_250000": clean_only.groupby("county")["amount"].sum(),
}).fillna(0)
compare["difference_pct"] = ((compare["all_rows"] - compare["without_imputed_and_250000"]) / compare["all_rows"] * 100).round(1)
display(compare)

,all_rows,without_imputed_and_250000,difference_pct
county,,,
Kiambu,411761.0,160011.0,61.1
Kisumu,512851.0,255689.0,50.1
Mombasa,517688.0,260472.0,49.7
Nairobi,705630.0,441414.0,37.4
Nakuru,513513.0,258155.0,49.7
Unknown,27156.0,27156.0,0.0


In [20]:
weekly = clean_df.copy()
weekly["week_start"] = weekly["txn_date"].dt.to_period("W-SUN").dt.start_time.dt.date
pivot_weekly = weekly.pivot_table(
    index="week_start", columns="county", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total",
).round(0)
pivot_weekly

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2771.0,0.0,18469.0
2026-03-02,18317.0,287933.0,285697.0,68967.0,56574.0,0.0,717488.0
2026-03-09,267745.0,23186.0,30505.0,52888.0,28730.0,0.0,403054.0
2026-03-16,22054.0,28126.0,25172.0,43135.0,39795.0,1639.0,159921.0
2026-03-23,11504.0,27077.0,17050.0,41401.0,18972.0,0.0,116004.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,47120.0,56814.0,276609.0,9824.0,420738.0
2026-04-13,12265.0,29471.0,26730.0,56406.0,26180.0,0.0,151052.0
2026-04-20,24507.0,37439.0,16687.0,65134.0,30862.0,1877.0,176506.0


**Findings (weekly pivot).** Weekly value runs from about KES 113,000 to KES 177,000 in ordinary weeks, with Nairobi typically contributing the largest share (KES 31,000 to 69,000 a week). Five cells stand far above the rest, from KES 267,745 to KES 287,933, and **each one is a KES 250,000 transaction**: Kisumu and Mombasa (week of 2 March), Kiambu (9 March), Nairobi (30 March) and Nakuru (6 April). The week of 2 March totals KES 717,488, about 4 times a typical week, entirely because of two such transactions. The first week (starting 23 February) holds only 1 March, and the last (27 April) holds only three days, so both are partial and should not be compared with full weeks. The weekly grand total is KES 2,688,599, matching the county table.

## Data Quality Report: Agent Transaction Extract

**Scope.** 612 raw transactions reduced to **600** clean rows, from 20 registered agents in five counties, dated 1 March to 29 April 2026. Counts come from the cleaning log.

| Dimension | Issue found | Rows | Treatment |
|---|---|---|---|
| Completeness | Missing `amount` (blank or the text "N/A") | 22 raw, 20 after deduplication | Filled with the median of the same transaction type (KES 1,804 for the 13 deposits, KES 1,750 for the 7 withdrawals) and flagged `amount_imputed`. The 20 filled amounts total KES 35,702, which is 1.3% of total value.|
| Uniqueness | Repeated `txn_id` | 12 | One row kept per `txn_id`, preferring the copy whose agent is registered |
| Consistency | `txn_type` in five spellings (case, trailing spaces); dates written with slashes | 384; 5 | Standardised to Deposit and Withdrawal; converted to ISO dates (YYYY-MM-DD) |
| Validity | Agent ID not in the register (AG099); text in `amount` | 8 raw, 7 after deduplication | Kept, county set to "Unknown", flagged `unknown_agent`; text converted to missing, then imputed |
| Accuracy | Five amounts of exactly KES 250,000; 40 IQR outliers in total | 5; 40 | Flagged `is_outlier`, not removed; verification recommended |
| Timeliness | Dates outside the expected range, before an agent's onboarding, or unparsable | 0 | None needed. The first and last weeks are partial |

**Bottom line.** The cleaned file is usable for county and weekly analysis. The main risk is accuracy: five unverified amounts make up 46.5% of value. County rankings hold with or without them, but totals and averages do not, so both views should be reported until the five transactions are confirmed.

**Privacy (Kenya Data Protection Act 2019).** This extract contains no customer data, but if it also held customer phone numbers, the **phone number** would be a direct identifier and therefore personal data, along with any name or national ID. The combination of phone number with date, amount and agent or county would also reveal a person's financial behaviour and location. I would protect it by: (1) data minimisation, dropping the phone number unless the analysis truly needs it; (2) pseudonymisation, replacing it with a salted hash or token, with the key stored separately under restricted access; (3) reporting only aggregates at county or agent level; (4) encryption in storage and transfer, role-based access, and keeping raw extracts out of GitHub, as this repository's `.gitignore` already does; and (5) governance steps: a documented lawful basis and purpose, a data protection impact assessment given the volume of financial data, registration with the Office of the Data Protection Commissioner, a breach-response process, and care over cross-border transfer when using cloud tools such as Colab. Real customer data should be processed only in an approved environment.

In [21]:
import os
os.makedirs("outputs", exist_ok=True)
export = clean_df.copy()
export["txn_date"] = export["txn_date"].dt.strftime("%Y-%m-%d")
export.to_csv("outputs/agent_transactions_clean.csv", index=False)

check = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Shape:", check.shape)
print("Duplicate txn_id:", check["txn_id"].duplicated().sum())
print("txn_type values:", sorted(check["txn_type"].unique()))
print("Columns:", check.columns.tolist())

from google.colab import files
files.download("outputs/agent_transactions_clean.csv")

Shape: (600, 9)
Duplicate txn_id: 0
txn_type values: ['Deposit', 'Withdrawal']
Columns: ['txn_id', 'agent_id', 'county', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'unknown_agent']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
imp = clean_df[clean_df["amount_imputed"]]
print("Imputed rows:", len(imp))
print("Total imputed value (KES):", f"{imp['amount'].sum():,.0f}")
print(f"Share of total value: {imp['amount'].sum() / clean_df['amount'].sum() * 100:.1f}%")
print(imp.groupby("txn_type")["amount"].agg(["count", "first"]))

Imputed rows: 20
Total imputed value (KES): 35,702
Share of total value: 1.3%
            count   first
txn_type                 
Deposit        13  1804.0
Withdrawal      7  1750.0
